# Kernels and carryover

A dose-response kernel is an *expression builder*: it declares its parameters — **scales**
carrying dose or outcome dimensions, **shapes** dimensionless — and returns `axiom.core` nodes
for the response, the dimensionless saturation, and the closed-form derivative. Carryover is a
normalized FIR weight vector, also built from nodes, applied with `Convolve`. Nothing here is a
numpy transform: every number comes from the interpreters (rule 3, "one `forward()`").

In [ ]:
import numpy as np

from axiom.core import D, Data, dimension, dimensionless, latex, value
from axiom.surface import (
    CARRYOVERS, KERNELS, AnyCarryover, AnyKernel, CarryoverKernel, CarryoverRole, DelayedCarryover,
    ExponentialKernel, GeometricCarryover, HillKernel, KernelRole, LinearKernel, LogisticKernel,
    NoCarryover, PowerKernel, ResponseKernel, WeibullCarryover, carryover_from_name, kernel_from_name,
)

In [ ]:
print(sorted(KERNELS), "|", sorted(CARRYOVERS))
hill: ResponseKernel = HillKernel(reference_dose=50.0, amplitude_scale=10.0)
dose = Data(name="a", dimension=D.currency)
for p in hill.parameters("a", D.currency, D.outcome):
    role: KernelRole = hill.roles[p.name.rsplit("_", 1)[0]] if p.name.rsplit("_", 1)[0] in hill.roles else "shape"
    print(f"{p.name:8s} dim={p.dimension!s:4s} prior={p.prior.family}{p.prior.hyper}")
print("saturating:", hill.saturating, "| roles:", dict(hill.roles))

In [ ]:
resp = hill.response(dose, "a")
print(dimension(resp), "|", dimension(hill.saturation(dose, "a")), "|", dimension(hill.derivative(dose, "a")))
print(latex(hill.saturation(dose, "a")))
theta = {"k_a": 50.0, "s_a": 2.0, "beta_a": 10.0}
grid = {"a": np.array([0.0, 25.0, 50.0, 100.0, 200.0])}
print(value(resp, data=grid, params=theta).round(3))
print(value(hill.derivative(dose, "a"), data=grid, params=theta).round(4))

## Unit invariance (exit criterion 6)

Express the same world in another unit: doses and the scale `k` multiplied by the same factor,
shapes untouched. The response is unchanged. This is what lets shape parameters pool across
studies in different currencies or time grids.

In [ ]:
cents = {"a": grid["a"] * 100}
theta_cents = {**theta, "k_a": theta["k_a"] * 100}
print(np.allclose(value(resp, data=cents, params=theta_cents), value(resp, data=grid, params=theta)))

## The families

In [ ]:
for fam in (LogisticKernel(reference_dose=50.0), ExponentialKernel(reference_dose=50.0), PowerKernel(reference_dose=50.0), LinearKernel(reference_dose=50.0)):
    k: AnyKernel = fam
    names = [p.name for p in k.parameters("a", D.currency, D.outcome)]
    print(f"{k.name:12s} saturating={k.saturating!s:5s} params={names}")
print(kernel_from_name("hill", reference_dose=10.0))

## Carryover

Weights are a dimensionless expression (`Pow`, `Reduce(keepdims=True)`, `Div`), so the jax
interpreter sees them and NUTS can sample their parameters. A unit impulse through `apply`
reproduces the weights; `half_life` is a diagnostic on parameter values.

In [ ]:
geo: CarryoverKernel = GeometricCarryover(max_lag=6)
w = geo.weights("a")
print(dimension(w), [p.name for p in geo.parameters("a")])
impulse = {"a": np.array([1.0, 0, 0, 0, 0, 0, 0, 0])}
print(value(geo.apply(dose, "a"), data=impulse, params={"lam_a": 0.6}).round(4))
print("half-life:", geo.half_life({"lam_a": 0.6}, "a"))
role: CarryoverRole = "shape"

In [ ]:
for c in (DelayedCarryover(max_lag=6), WeibullCarryover(max_lag=6), NoCarryover()):
    cc: AnyCarryover = c
    print(cc.name, [p.name for p in cc.parameters("a")])
delayed = DelayedCarryover(max_lag=8)
print(value(delayed.weights("a"), params={"lam_a": 0.5, "theta_a": 3.0}).round(3))
print(carryover_from_name("geometric", max_lag=4))

Draw-shaped parameters broadcast: a `(draws, 1)` array of `lam` gives one normalized weight
row per draw — the normalization keeps the lag axis (`Reduce(keepdims=True)`).

In [ ]:
value(geo.weights("a"), params={"lam_a": np.array([[0.2], [0.5], [0.9]])}).round(3)